In [4]:
import os
import pypdf
from pypdf import PdfReader
from collections import defaultdict

In [8]:
path_x = 'C:/Users/arj26323/OneDrive - University of Georgia/Documents/Manuscripts/Network analysis/Financials for funding entities'

In [38]:
import os
import re

def extract_pdf_text(pdf_path):
    """Extracts text from all pages of a PDF."""
    text = ""
    try:
        reader = PdfReader(pdf_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
    return text

# def extract_organization_name(pdf_path):
#     """
#     Extract organization name from the first page of an IRS Form 990.
#     """
#     try:
#         reader = PdfReader(pdf_path)

#         if len(reader.pages) == 0:
#             return "Unknown Organization"

#         text = reader.pages[0].extract_text()

#         lines = [line.strip() for line in text.splitlines() if line.strip()]

#         for i, line in enumerate(lines):
#             normalized = re.sub(r"\s+", " ", line.lower())

#             if normalized == "name of organization":
#                 # Return the nearest non-empty line above
#                 if i > 0:
#                     return lines[i-1]

#         return "Unknown Organization"

#     except Exception as e:
#         print(f"Error extracting organization name: {e}")
#         return "Unknown Organization"

def scan_pdf_folder(root_folder, target_keywords):
    """
    Scans a folder and subfolders for PDFs containing target keywords,
    identifies PDFs sharing keywords, and groups PDFs by organization name.
    """
    pdf_data = {}
    keyword_to_paths = defaultdict(set)
    org_to_paths = defaultdict(set)
    
    target_keywords = [kw.lower() for kw in target_keywords]

    # 1. Walk through the directory and extract text
    print("Scanning folders and extracting text from PDFs...")
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.lower().endswith('.pdf'):
                pdf_path = os.path.join(dirpath, filename)
               
                relative_path = os.path.relpath(pdf_path, root_folder)
                
                raw_text = extract_pdf_text(pdf_path)
                if not raw_text.strip():
                    continue
                
                lower_text = raw_text.lower()
                
                # Extract Organization Name using Regex
                org_name = extract_organization_name(pdf_path)
                org_to_paths[org_name].add(relative_path)
                
                # Task A: Check target keywords
                found_keywords = set()
                for kw in target_keywords:
                    if kw in lower_text:
                        found_keywords.add(kw)
                        keyword_to_paths[kw].add(relative_path)
                
                pdf_data[relative_path] = {
                    "full_path": pdf_path,
                    "organization": org_name,
                    "found_keywords": found_keywords
                }

    # # --- DEBUG: PRINT ALL EXTRACTED ORGANIZATION NAMES ---
    # print("\n" + "="*60)
    # print("DEBUG: All Extracted Organization Names per PDF")
    # print("="*60)
    # for path, data in sorted(pdf_data.items()):
    #     print(f"File: {path}")
    #     print(f"  -> Org : {data['organization']}")
    
    # # --- RESULTS FOR TASK A ---
    # print("\n" + "="*60)
    # print("TASK A: PDFs containing specific keywords")
    # print("="*60)
    # for kw in target_keywords:
    #     matching_paths = keyword_to_paths.get(kw, set())
    #     print(f"Keyword '{kw}':")
    #     if matching_paths:
    #         for path in sorted(matching_paths):
    #             print(f"  - {path}")
    #     else:
    #         print("  - None found")

    # --- RESULTS FOR TASK B ---
    print("\n" + "="*60)
    print("TASK B: PDFs that share keywords (Intersection)")
    print("="*60)
    
    paths = list(pdf_data.keys())
    shared_results = []
    
    for i in range(len(paths)):
        for j in range(i + 1, len(paths)):
            path1 = paths[i]
            path2 = paths[j]
            
            set1 = pdf_data[path1]["found_keywords"]
            set2 = pdf_data[path2]["found_keywords"]
            
            common_keywords = set1.intersection(set2)
            if common_keywords:
                shared_results.append((path1, path2, common_keywords))

    if shared_results:
        for p1, p2, common in shared_results:
            kw_list = ", ".join(common)
            print(f"• '{p1}' and '{p2}' share keywords: [{kw_list}]")
    else:
        print("No PDFs shared the same target keywords from your list.")

    # # --- RESULTS FOR TASK C ---
    # print("\n" + "="*60)
    # print("TASK C: PDFs sharing the same Organization Name")
    # print("="*60)
    
    # shared_orgs_found = False
    # for org, file_paths in org_to_paths.items():
    #     if len(file_paths) > 1 and org != "Unknown Organization":
    #         shared_orgs_found = True
    #         print(f"Organization: '{org}'")
    #         for path in sorted(file_paths):
    #             print(f"  - {path}")
    #         print()

    # if not shared_orgs_found:
    #     print("No multiple PDFs shared the exact same organization name.")

# ==========================================
# CONFIGURATION & EXECUTION
# ==========================================
if __name__ == "__main__":
    ROOT_DIRECTORY = path_x 
    KEYWORDS = ["marsh", "salt marsh", "oyster", "restoration", "living shoreline", "oyster reef", "coastal conservation association", 
                "friends of the savannah", "georgia conservancy", "georgia alabama land trust", "glynn environmental coalition", 
                "jekyll island foundation", "manomet", "ogeechee riverkeeper", "one hundred miles", "ossabaw island foundation",
               "satilla riverkeeper", "savannah riverkeeper", "st. mary", "st. simon", "the nature conservancy", "tybee island marine science",
               "wormsloe"]
    
    if os.path.exists(ROOT_DIRECTORY):
        scan_pdf_folder(ROOT_DIRECTORY, KEYWORDS)
    else:
        print(f"Directory '{ROOT_DIRECTORY}' does not exist. Please update the path.")

Scanning folders and extracting text from PDFs...

TASK B: PDFs that share keywords (Intersection)
• '2024_Georgia Power Foundation Inc In Receivership - Full Filing - Nonprofit Explorer - ProPublica.pdf' and 'Coastal Conservation Association\2024.pdf' share keywords: [restoration]
• '2024_Georgia Power Foundation Inc In Receivership - Full Filing - Nonprofit Explorer - ProPublica.pdf' and 'GA conservancy\2025_filing_Georgia Conservancy Inc - Full Filing - Nonprofit Explorer - ProPublica.pdf' share keywords: [marsh, georgia conservancy, restoration]
• '2024_Georgia Power Foundation Inc In Receivership - Full Filing - Nonprofit Explorer - ProPublica.pdf' and 'Gerogia_Alabama Land Trust\2024.pdf' share keywords: [restoration]
• '2024_Georgia Power Foundation Inc In Receivership - Full Filing - Nonprofit Explorer - ProPublica.pdf' and 'One Hundred Miles\2024.pdf' share keywords: [marsh]
• '2024_Georgia Power Foundation Inc In Receivership - Full Filing - Nonprofit Explorer - ProPublica.pd

In [10]:






'''
def extract_pdf_text(pdf_path):
    """Extracts text from all pages of a PDF."""
    text = ""
    try:
        reader = PdfReader(pdf_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
    return text.lower()

def scan_pdf_folder(root_folder, target_keywords):
    """
    Scans a folder and subfolders for PDFs containing target keywords,
    identifies PDFs that share keywords, and displays relative paths/subfolders.
    """
    pdf_data = {}
    keyword_to_paths = defaultdict(set)
    
    # Normalize target keywords to lowercase
    target_keywords = [kw.lower() for kw in target_keywords]

    # 1. Walk through the directory and extract text
    print("Scanning folders and extracting text from PDFs...")
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.lower().endswith('.pdf'):
                pdf_path = os.path.join(dirpath, filename)
                
                # Get path relative to the root folder (keeps subfolder structure clean)
                relative_path = os.path.relpath(pdf_path, root_folder)
                
                text = extract_pdf_text(pdf_path)
                
                if not text.strip():
                    continue
                
                # Task A: Check which target keywords are present in this PDF
                found_keywords = set()
                for kw in target_keywords:
                    if kw in text:
                        found_keywords.add(kw)
                        keyword_to_paths[kw].add(relative_path)
                
                pdf_data[relative_path] = {
                    "full_path": pdf_path,
                    "text": text,
                    "found_keywords": found_keywords
                }

    # --- RESULTS FOR TASK A ---
    print("\n" + "="*60)
    print("TASK A: PDFs containing specific keywords (with subfolders)")
    print("="*60)
    for kw in target_keywords:
        matching_paths = keyword_to_paths.get(kw, set())
        print(f"Keyword '{kw}':")
        if matching_paths:
            for path in sorted(matching_paths):
                print(f"  - {path}")
        else:
            print("  - None found")

    # --- RESULTS FOR TASK B ---
    print("\n" + "="*60)
    print("TASK B: PDFs that share keywords (Intersection)")
    print("="*60)
    
    paths = list(pdf_data.keys())
    shared_results = []
    
    # Compare each unique pair of PDFs based on their found target keywords
    for i in range(len(paths)):
        for j in range(i + 1, len(paths)):
            path1 = paths[i]
            path2 = paths[j]
            
            set1 = pdf_data[path1]["found_keywords"]
            set2 = pdf_data[path2]["found_keywords"]
            
            common_keywords = set1.intersection(set2)
            if common_keywords:
                shared_results.append((path1, path2, common_keywords))

    if shared_results:
        for p1, p2, common in shared_results:
            kw_list = ", ".join(common)
            print(f"• '{p1}' and '{p2}' share keywords: [{kw_list}]")
    else:
        print("No PDFs shared the same target keywords from your list.")

# ==========================================
# CONFIGURATION & EXECUTION
# ==========================================
if __name__ == "__main__":
    # Replace with the path to your root folder
    ROOT_DIRECTORY = path_x 
    
    # Define the set of keywords you want to look for
    KEYWORDS = ["marsh", "salt marsh", "oyster", "restoration", "living shoreline", "oyster reef"]
    
    if os.path.exists(ROOT_DIRECTORY):
        scan_pdf_folder(ROOT_DIRECTORY, KEYWORDS)
    else:
        print(f"Directory '{ROOT_DIRECTORY}' does not exist. Please update the path.")'''




'''def extract_pdf_text(pdf_path):
    """Extracts text from all pages of a PDF."""
    text = ""
    try:
        reader = PdfReader(pdf_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
    return text.lower()

def scan_pdf_folder(root_folder, target_keywords):
    """
    Scans a folder and subfolders for PDFs containing target keywords,
    and identifies PDFs that share keywords.
    """
    pdf_data = {}
    keyword_to_pdfs = defaultdict(set)
    
    # Normalize target keywords to lowercase
    target_keywords = [kw.lower() for kw in target_keywords]

    # 1. Walk through the directory and extract text
    print("Scanning folders and extracting text from PDFs...")
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.lower().endswith('.pdf'):
                pdf_path = os.path.join(dirpath, filename)
                text = extract_pdf_text(pdf_path)
                
                if not text.strip():
                    continue
                
                # Task A: Check which target keywords are present in this PDF
                found_keywords = set()
                for kw in target_keywords:
                    if kw in text:
                        found_keywords.add(kw)
                        keyword_to_pdfs[kw].add(filename)
                
                pdf_data[filename] = {
                    "path": pdf_path,
                    "text": text,
                    "found_keywords": found_keywords
                }

    # --- RESULTS FOR TASK A ---
    print("\n" + "="*50)
    print("TASK A: PDFs containing specific keywords")
    print("="*50)
    for kw in target_keywords:
        matching_files = keyword_to_pdfs.get(kw, set())
        print(f"Keyword '{kw}':")
        if matching_files:
            for f in matching_files:
                print(f"  - {f}")
        else:
            print("  - None found")

    # --- RESULTS FOR TASK B ---
    print("\n" + "="*50)
    print("TASK B: PDFs that share keywords (Intersection)")
    print("="*50)
    
    filenames = list(pdf_data.keys())
    shared_results = []
    
    # Compare each unique pair of PDFs based on their found target keywords
    for i in range(len(filenames)):
        for j in range(i + 1, len(filenames)):
            file1 = filenames[i]
            file2 = filenames[j]
            
            set1 = pdf_data[file1]["found_keywords"]
            set2 = pdf_data[file2]["found_keywords"]
            
            common_keywords = set1.intersection(set2)
            if common_keywords:
                shared_results.append((file1, file2, common_keywords))

    if shared_results:
        for f1, f2, common in shared_results:
            print(f"• '{f1}' and '{f2}' share keywords: {', ' + str(list(common)}")
    else:
        print("No PDFs shared the same target keywords from your list.")

# ==========================================
# CONFIGURATION & EXECUTION
# ==========================================
if __name__ == "__main__":
    # Replace with the path to your root folder
    ROOT_DIRECTORY = path_x 
    
    # Define the set of keywords you want to look for
    KEYWORDS = ["marsh", "salt marsh", "oyster", "restoration", "living shoreline", "oyster reef"]
    
    if os.path.exists(ROOT_DIRECTORY):
        scan_pdf_folder(ROOT_DIRECTORY, KEYWORDS)
    else:
        print(f"Directory '{ROOT_DIRECTORY}' does not exist. Please update the path.")'''

Scanning folders and extracting text from PDFs...

TASK A: PDFs containing specific keywords
Keyword 'marsh':
  - FY 2026 Governor's Report - FINAL.pdf
  - 2024.pdf
  - 2024_Georgia Power Foundation Inc In Receivership - Full Filing - Nonprofit Explorer - ProPublica.pdf
  - 2025_filing_Georgia Conservancy Inc - Full Filing - Nonprofit Explorer - ProPublica.pdf
  - FY 2025 Governor's Budget Report - FINAL.pdf
  - FY 2024 Governor's Report (FINAL).pdf
Keyword 'salt marsh':
  - 2025_filing_Georgia Conservancy Inc - Full Filing - Nonprofit Explorer - ProPublica.pdf
Keyword 'oyster':
  - 2023.pdf
  - 2024.pdf
  - 2025.pdf
  - 2025_filing_Georgia Conservancy Inc - Full Filing - Nonprofit Explorer - ProPublica.pdf
Keyword 'restoration':
  - FY 2026 Governor's Report - FINAL.pdf
  - 2024.pdf
  - 2024_Georgia Power Foundation Inc In Receivership - Full Filing - Nonprofit Explorer - ProPublica.pdf
  - 2025_filing_Georgia Conservancy Inc - Full Filing - Nonprofit Explorer - ProPublica.pdf
  - FY 